# ECOS — Block 3: Experimental Control and Validation

**Objective:** Verify that measurements are trustworthy before drawing scientific
conclusions. Every reviewer will ask: *"Are your results real, or are they artifacts
of your measurement setup?"*

This block addresses:
- Is temperature well controlled? Does it confound $C_l$?
- Is the density measurement reliable? (vessel radius sensitivity)
- Are there outliers that distort the analysis?
- What fraction of $C_l$ variance is explained by each factor?

> **This block does not generate paper results directly.** It generates **confidence**
> in the results of Blocks 1 and 2, and identifies caveats to report.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from IPython.display import display

sys.path.insert(0, str(Path().resolve()))
import ecos_loader

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

base_dir = Path('../database')
df = ecos_loader.build_catalog(base_dir)

for col in ('pva_pct', 'pg_pct', 'cycle'):
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['pva_real'] = df['pva_pct'].apply(lambda x: x / 10 if x > 100 else x)

# Derived temperature column used throughout this block
if 'US_T1' in df.columns and 'US_T2' in df.columns:
    df['US_T_mean'] = (df['US_T1'] + df['US_T2']) / 2
else:
    df['US_T_mean'] = np.nan

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

# Color convention (Study Rules §7): tab10, consistent with Blocks 1 and 2
_cmap      = plt.cm.tab10
PVA_LEVELS = sorted(df['pva_real'].dropna().unique())
PG_LEVELS  = sorted(df['pg_pct'].dropna().unique())
PVA_COLOR  = {pva: _cmap(i) for i, pva in enumerate(PVA_LEVELS)}
PG_COLOR   = {pg:  _cmap(i) for i, pg  in enumerate(PG_LEVELS)}
PG_MARKER  = {pg: m for pg, m in zip(PG_LEVELS, ['o', 's', '^', 'D'])}

sorted_cycles = sorted(df['cycle'].dropna().unique())
CYCLE_COLOR   = {c: _cmap(i) for i, c in enumerate(sorted_cycles)}

conditions = (
    df[['pva_real', 'pg_pct']].drop_duplicates()
    .sort_values(['pva_real', 'pg_pct']).reset_index(drop=True)
)
labels_cond = [f'PVA {r.pva_real:.4g}%\nPG {int(r.pg_pct)}%' for _, r in conditions.iterrows()]
colors_box  = [PVA_COLOR[r.pva_real] for _, r in conditions.iterrows()]

print(f'Catalog shape: {df.shape}')
print(f'Cycles: {[int(c) for c in sorted_cycles]}')
print(f'PVA levels: {PVA_LEVELS}   PG levels: {PG_LEVELS}')
print(f'US_T_mean available: {df["US_T_mean"].notna().sum()} rows')


---
## Cell 3.0 — Temperature control overview

Left: distribution of US temperatures (mean of T1, T2). Target = 36°C.
Right: distribution of density measurement temperatures (T_C).

**Red flags:** US SD > 1°C (poor control); any measurement outside 30–40°C;
systematic T1 ≠ T2 (sensor calibration issue — tested in Cell 3.1).

> A 1°C error in water temperature causes ≈ 3 m/s error in $C_w$,
> which propagates directly into $C_l$. Temperature control quality
> sets a floor on measurement precision.

In [ ]:
# --- Cell 3.0: Temperature control overview ---
T_US_TARGET = 36.0

T_us   = df['US_T_mean'].dropna()
T_dens = df['DENS_T_C'].dropna() if 'DENS_T_C' in df.columns else pd.Series(dtype=float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: US temperature ---
if len(T_us) > 0:
    ax1.hist(T_us, bins=20, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.5)
    ax1.axvline(T_US_TARGET, color='red', linestyle='--', linewidth=1.5, label=f'Target = {T_US_TARGET}°C')
    ax1.axvline(T_us.mean(), color='k', linestyle='-', linewidth=1.5, label=f'Mean = {T_us.mean():.2f}°C')
    pct_ok = ((T_us - T_US_TARGET).abs() <= 1).mean() * 100
    stats_us = (f'Mean = {T_us.mean():.2f} °C   SD = {T_us.std():.2f} °C\n'
                f'Range [{T_us.min():.1f}, {T_us.max():.1f}] °C\n'
                f'{pct_ok:.1f}% within ±1°C of target')
    ax1.text(0.02, 0.97, stats_us, transform=ax1.transAxes, fontsize=9,
             va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax1.set_xlabel('US temperature (°C)', fontsize=12)
    ax1.set_ylabel('Count', fontsize=12)
    ax1.set_title(f'US temperature distribution  (n={len(T_us)})', fontsize=13)
    ax1.legend(fontsize=9)
else:
    ax1.text(0.5, 0.5, 'US_T_mean not available', ha='center', transform=ax1.transAxes)

# --- Right: DENS temperature ---
if len(T_dens) > 0:
    ax2.hist(T_dens, bins=20, color='coral', alpha=0.7, edgecolor='white', linewidth=0.5)
    ax2.axvline(T_dens.mean(), color='k', linestyle='-', linewidth=1.5, label=f'Mean = {T_dens.mean():.2f}°C')
    stats_dens = (f'Mean = {T_dens.mean():.2f} °C   SD = {T_dens.std():.2f} °C\n'
                  f'Range [{T_dens.min():.1f}, {T_dens.max():.1f}] °C')
    ax2.text(0.02, 0.97, stats_dens, transform=ax2.transAxes, fontsize=9,
             va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax2.set_xlabel('DENS temperature (°C)', fontsize=12)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title(f'Density measurement temperature  (n={len(T_dens)})', fontsize=13)
    ax2.legend(fontsize=9)
else:
    ax2.text(0.5, 0.5, 'DENS_T_C not available', ha='center', transform=ax2.transAxes)

plt.tight_layout()
plt.savefig(fig_dir / 'B3_00_temperature_overview.png', dpi=150, bbox_inches='tight')
plt.show()

# Red flags
print('Red flags:')
found = False
if len(T_us) > 0:
    if T_us.std() > 1:
        print(f'  [!] US temperature SD = {T_us.std():.3f}°C > 1°C')
        found = True
    n_anom = int(((T_us < 30) | (T_us > 40)).sum())
    if n_anom:
        print(f'  [!] {n_anom} US measurement(s) outside 30-40°C range')
        found = True
    if not found:
        print(f'  None. SD = {T_us.std():.3f}°C, {pct_ok:.1f}% within ±1°C of target.')


---
## Cell 3.1 — T1 vs T2 sensor agreement (Bland-Altman)

Left: T1 vs T2 scatter with identity line, colored by cycle.
Right: Bland-Altman plot — difference (T1 − T2) vs mean. Limits of agreement = mean ± 1.96 SD.

If mean(T1 − T2) ≠ 0 significantly → one sensor has a systematic bias;
the $C_w$ calculation is affected by a known offset. Quantify and report.

In [ ]:
# --- Cell 3.1: T1 vs T2 sensor agreement and Bland-Altman ---
if 'US_T1' not in df.columns or 'US_T2' not in df.columns:
    print('US_T1 / US_T2 not available.')
else:
    mask_t = df['US_T1'].notna() & df['US_T2'].notna()
    T1   = df.loc[mask_t, 'US_T1']
    T2   = df.loc[mask_t, 'US_T2']
    diff = T1 - T2
    tmean = (T1 + T2) / 2

    mean_diff = diff.mean()
    sd_diff   = diff.std(ddof=1)
    loa_lo    = mean_diff - 1.96 * sd_diff
    loa_hi    = mean_diff + 1.96 * sd_diff
    t_stat, p_t = stats.ttest_1samp(diff, 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: T1 vs T2 scatter ---
    for cyc in sorted_cycles:
        mask_c = (df['cycle'] == cyc) & mask_t
        ax1.scatter(df.loc[mask_c, 'US_T1'], df.loc[mask_c, 'US_T2'],
                    color=CYCLE_COLOR[cyc], s=30, alpha=0.8,
                    edgecolors='white', linewidths=0.4,
                    label=f'Cycle {int(cyc)}')

    t_lo = min(T1.min(), T2.min()) - 0.3
    t_hi = max(T1.max(), T2.max()) + 0.3
    ax1.plot([t_lo, t_hi], [t_lo, t_hi], 'k--', alpha=0.5, linewidth=1.2, label='Identity T1=T2')
    ax1.set_xlim(t_lo, t_hi)
    ax1.set_ylim(t_lo, t_hi)
    ax1.set_xlabel('T1 (°C)', fontsize=12)
    ax1.set_ylabel('T2 (°C)', fontsize=12)
    ax1.set_title('T1 vs T2 sensor agreement', fontsize=13)
    ax1.legend(fontsize=8, framealpha=0.9)
    ax1.set_aspect('equal', adjustable='box')

    # --- Right: Bland-Altman ---
    ax2.scatter(tmean, diff, color='steelblue', s=30, alpha=0.8,
                edgecolors='white', linewidths=0.4)
    x_ba = [tmean.min() - 0.2, tmean.max() + 0.2]
    ax2.hlines(mean_diff, x_ba[0], x_ba[1], colors='k',    linestyles='-',  linewidth=1.5,
               label=f'Mean diff = {mean_diff:.3f}°C')
    ax2.hlines(loa_hi,    x_ba[0], x_ba[1], colors='red',  linestyles='--', linewidth=1.2,
               label=f'+1.96 SD = {loa_hi:.3f}°C')
    ax2.hlines(loa_lo,    x_ba[0], x_ba[1], colors='red',  linestyles='--', linewidth=1.2,
               label=f'-1.96 SD = {loa_lo:.3f}°C')
    ax2.hlines(0,         x_ba[0], x_ba[1], colors='grey', linestyles=':',  linewidth=1.0, alpha=0.6)
    ax2.fill_between(x_ba, loa_lo, loa_hi, alpha=0.07, color='red')
    ax2.set_xlim(x_ba)
    ax2.set_xlabel('Mean (T1+T2)/2 (°C)', fontsize=12)
    ax2.set_ylabel('T1 - T2 (°C)', fontsize=12)
    ax2.set_title('Bland-Altman: T1 vs T2 agreement', fontsize=13)
    ax2.legend(fontsize=8, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(fig_dir / 'B3_01_T1_T2_agreement.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Sensor agreement (n={len(diff)}):')
    print(f'  Mean(T1-T2) = {mean_diff:.4f}°C   SD = {sd_diff:.4f}°C')
    print(f'  Limits of agreement: [{loa_lo:.3f}, {loa_hi:.3f}]°C  (±1.96 SD)')
    print(f'  Paired t-test (H0: mean diff=0): t = {t_stat:.3f}, p = {p_t:.4f}')
    flag_bias = ' [!] Significant sensor bias' if p_t < 0.05 else ' OK (no significant bias)'
    print(f'  Conclusion:{flag_bias}')
    if abs(mean_diff) > 0:
        cw_bias = abs(mean_diff) / 2 * 3  # approx 3 m/s per 1°C in Cw
        print(f'  Estimated Cw bias from sensor offset: ~{cw_bias:.2f} m/s (half the T offset × 3 m/s/°C)')


---
## Cell 3.2 — $C_l$ vs temperature: confounding check

Scatter: x = $T_{\text{mean}}$, y = $C_l$, color = PVA%. Regression line + statistics.

**Expected result:** r ≈ 0, p > 0.05 — temperature is NOT confounding.
**Concerning result:** significant correlation — temperature explains part of $C_l$
variability, potentially masking real composition effects.

> Correlation ≠ causation. Even a significant r may be spurious (both vary by day).
> Causation requires a controlled temperature sweep of the same specimen — future work.

In [ ]:
# --- Cell 3.2: Cl vs temperature — confounding check ---
mask_ct = df['US_Cl'].notna() & df['US_T_mean'].notna()
if mask_ct.sum() < 3:
    print('Insufficient data for Cl vs T analysis.')
else:
    T_ct  = df.loc[mask_ct, 'US_T_mean']
    Cl_ct = df.loc[mask_ct, 'US_Cl']

    r_val, p_val = stats.pearsonr(T_ct, Cl_ct)
    slope, intercept = np.polyfit(T_ct, Cl_ct, 1)
    R2 = r_val ** 2

    fig, ax = plt.subplots(figsize=(7, 5))

    for pva in PVA_LEVELS:
        sub = df[mask_ct & (df['pva_real'] == pva)]
        ax.scatter(sub['US_T_mean'], sub['US_Cl'],
                   color=PVA_COLOR[pva], s=35, alpha=0.8,
                   edgecolors='white', linewidths=0.4,
                   label=f'PVA {pva:.4g}%')

    T_rng = np.linspace(T_ct.min(), T_ct.max(), 100)
    ax.plot(T_rng, slope * T_rng + intercept, 'k--', alpha=0.5, linewidth=1.5, label='Regression')

    confound = 'CONCERN' if p_val < 0.05 else 'OK (not confounding)'
    stats_str = (f'r = {r_val:.3f}   p = {p_val:.4f}   R2 = {R2:.4f}\n'
                 f'Slope = {slope:.2f} m/s per °C\n'
                 f'Status: {confound}')
    ax.text(0.02, 0.05, stats_str, transform=ax.transAxes, fontsize=9, va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    ax.set_xlabel('US temperature (°C)', fontsize=12)
    ax.set_ylabel('Longitudinal velocity $C_l$ (m/s)', fontsize=12)
    ax.set_title('$C_l$ vs temperature — confounding check', fontsize=13)
    ax.legend(title='PVA %', fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.savefig(fig_dir / 'B3_02_Cl_vs_T.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Cl vs T: r={r_val:.4f}, p={p_val:.4f}, R2={R2:.4f}, slope={slope:.3f} m/s/°C')

    # Extended: partial correlation controlling for PVA (numeric)
    if p_val < 0.05:
        print('\n[Partial correlation: Cl vs T controlling for PVA% (numeric)]')
        mask_pva = mask_ct & df['pva_real'].notna()
        X_ctrl = np.column_stack([df.loc[mask_pva, 'pva_real'].values, np.ones(mask_pva.sum())])
        T_res  = df.loc[mask_pva, 'US_T_mean'].values  - X_ctrl @ np.linalg.lstsq(X_ctrl, df.loc[mask_pva, 'US_T_mean'].values,  rcond=None)[0]
        Cl_res = df.loc[mask_pva, 'US_Cl'].values      - X_ctrl @ np.linalg.lstsq(X_ctrl, df.loc[mask_pva, 'US_Cl'].values,      rcond=None)[0]
        r_part, p_part = stats.pearsonr(T_res, Cl_res)
        print(f'  Partial r (T | PVA%) = {r_part:.4f}, p = {p_part:.4f}')
        if abs(r_part) < abs(r_val) * 0.5:
            print('  -> Partial r << raw r: PVA% accounts for most of the T-Cl covariance.')
            print('     Temperature is likely NOT a genuine confounder.')
        else:
            print('  -> Partial r remains substantial: temperature may be a genuine confounder.')


---
## Cell 3.3 — Temperature distribution across conditions

Boxplot of US temperature grouped by condition (PVA% × PG%). If some conditions were
systematically measured at different temperatures, any $C_l$ differences between them
are partly confounded with temperature.

One-way ANOVA (and Kruskal-Wallis) tests whether mean temperature differs across conditions.

In [ ]:
# --- Cell 3.3: Temperature distribution by condition ---
rng = np.random.default_rng(42)
data_t_cond = [
    df[(df['pva_real'] == r.pva_real) & (df['pg_pct'] == r.pg_pct)]['US_T_mean'].dropna().values
    for _, r in conditions.iterrows()
]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(data_t_cond, labels=labels_cond, patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2},
                whiskerprops={'linewidth': 1.2}, capprops={'linewidth': 1.2})

for i, (patch, col) in enumerate(zip(bp['boxes'], colors_box)):
    patch.set_facecolor(col)
    patch.set_alpha(0.5)
    if len(data_t_cond[i]) > 0:
        x_pts = rng.uniform(-0.15, 0.15, size=len(data_t_cond[i])) + (i + 1)
        ax.scatter(x_pts, data_t_cond[i], color=col, s=30, zorder=5,
                   edgecolors='white', linewidths=0.5, alpha=0.9)

ax.set_xlabel('Condition', fontsize=12)
ax.set_ylabel('US temperature (°C)', fontsize=12)
ax.set_title('US temperature by condition — confound check\n(ideal: all boxes at the same level)', fontsize=12)
ax.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig(fig_dir / 'B3_03_T_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

# ANOVA: does T differ across conditions?
groups_t = [g for g in data_t_cond if len(g) >= 2]
if len(groups_t) >= 2:
    F_anova, p_anova = stats.f_oneway(*groups_t)
    H_kw,    p_kw    = stats.kruskal(*groups_t)
    print(f'One-way ANOVA (T across conditions): F = {F_anova:.3f}, p = {p_anova:.4f}')
    print(f'Kruskal-Wallis:                      H = {H_kw:.3f},   p = {p_kw:.4f}')
    if p_anova < 0.05:
        print('[!] Temperature differs significantly across conditions — potential confound in Cl comparisons.')
        try:
            from statsmodels.stats.multicomp import pairwise_tukeyhsd
            df_t3 = df[df['US_T_mean'].notna()].copy()
            df_t3['cond'] = df_t3['pva_real'].astype(str) + '_PG' + df_t3['pg_pct'].astype(str)
            tukey_t = pairwise_tukeyhsd(df_t3['US_T_mean'], df_t3['cond'], alpha=0.05)
            print('Tukey HSD (T by condition):')
            print(tukey_t.summary())
        except ImportError:
            print('  (statsmodels not available for post-hoc test)')
    else:
        print('Temperature is consistent across conditions — no confound detected.')


---
## Cell 3.4 — Temperature distribution across cycles

Boxplot of US temperature grouped by cycle. If later cycles were measured at higher
temperatures, the temporal $C_l$ trend could be contaminated by temperature drift.

In [ ]:
# --- Cell 3.4: Temperature distribution by cycle ---
rng4 = np.random.default_rng(44)
data_t_cyc = [
    df[df['cycle'] == c]['US_T_mean'].dropna().values
    for c in sorted_cycles
]
labels_cyc   = [f'Cycle {int(c)}' for c in sorted_cycles]
colors_cyc   = [CYCLE_COLOR[c] for c in sorted_cycles]

fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot(data_t_cyc, labels=labels_cyc, patch_artist=True,
                medianprops={'color': 'black', 'linewidth': 2},
                whiskerprops={'linewidth': 1.2}, capprops={'linewidth': 1.2})

for i, (patch, col) in enumerate(zip(bp['boxes'], colors_cyc)):
    patch.set_facecolor(col)
    patch.set_alpha(0.5)
    if len(data_t_cyc[i]) > 0:
        x_pts = rng4.uniform(-0.12, 0.12, size=len(data_t_cyc[i])) + (i + 1)
        ax.scatter(x_pts, data_t_cyc[i], color=col, s=30, zorder=5,
                   edgecolors='white', linewidths=0.5, alpha=0.9)

ax.set_xlabel('Cycle', fontsize=12)
ax.set_ylabel('US temperature (°C)', fontsize=12)
ax.set_title('US temperature by cycle — temporal confound check\n(ideal: boxes at the same level)', fontsize=12)

plt.tight_layout()
plt.savefig(fig_dir / 'B3_04_T_by_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

groups_t4 = [g for g in data_t_cyc if len(g) >= 2]
if len(groups_t4) >= 2:
    F4, p4 = stats.f_oneway(*groups_t4)
    H4, p4kw = stats.kruskal(*groups_t4)
    print(f'One-way ANOVA (T across cycles): F = {F4:.3f}, p = {p4:.4f}')
    print(f'Kruskal-Wallis:                  H = {H4:.3f}, p = {p4kw:.4f}')
    if p4 < 0.05:
        print('[!] Temperature differs significantly across cycles — cycle trend in Cl may be partly temperature-driven.')
    else:
        print('Temperature is consistent across cycles — no temporal temperature confound detected.')
    print('\nMean temperature per cycle:')
    for c, grp in zip(sorted_cycles, data_t_cyc):
        if len(grp) > 0:
            print(f'  Cycle {int(c)}: {np.mean(grp):.3f} ± {np.std(grp, ddof=1):.3f} °C  (n={len(grp)})')
else:
    print('Insufficient cycles for between-cycle temperature comparison.')


---
## Cell 3.5 — Density measurement sensitivity: vessel radius

Formula: $\rho = m / (\pi r^2 \Delta h)$. Since $r$ appears squared,
a 1% error in $r$ causes ≈ 2% error in $\rho$ (in the opposite direction):
$d\rho/\rho = -2\, dr/r$.

Left: family of $\rho$ vs $r$ curves (one per specimen). Green band = physically
expected range [1.00, 1.15] g/cm³. Vertical line = nominal radius.
Right: boxplots comparing original density vs recalculated at `r_test`.

**Adjust `r_test` to explore the impact of radius measurement uncertainty.**

In [ ]:
# --- Cell 3.5: Density sensitivity to vessel radius ---
# ========== USER: set test radius (cm) to compare against nominal ==========
r_test = 9.15

req_cols = ['DENS_r_vessel_cm', 'DENS_mass_g', 'DENS_delta_h_cm', 'DENS_density_gcm3']
if not all(c in df.columns for c in req_cols):
    print('Required density columns not available.')
else:
    mask5 = df[req_cols].notna().all(axis=1) & (df['DENS_delta_h_cm'] != 0)
    sub5  = df[mask5].copy()

    r_nom = sub5['DENS_r_vessel_cm'].median()  # nominal radius

    # Sensitivity: d(rho)/d(r) at nominal r = -2*rho/r
    rho_nom_mean = sub5['DENS_density_gcm3'].mean()
    sens = -2 * rho_nom_mean / r_nom
    rel_sens = -2  # dimensionless: (drho/rho) / (dr/r)

    radii = np.linspace(max(0.5, r_nom * 0.85), r_nom * 1.15, 200)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: rho vs r family of curves ---
    for pva in PVA_LEVELS:
        sub_pva = sub5[sub5['pva_real'] == pva]
        for _, row in sub_pva.iterrows():
            rho_curve = row['DENS_mass_g'] / (np.pi * radii ** 2 * row['DENS_delta_h_cm'])
            ax1.plot(radii, rho_curve, color=PVA_COLOR[pva], alpha=0.25, linewidth=0.7)

    # Legend dummy lines
    for pva in PVA_LEVELS:
        ax1.plot([], [], color=PVA_COLOR[pva], linewidth=2, label=f'PVA {pva:.4g}%')

    ax1.axvline(r_nom, color='k', linestyle='--', linewidth=1.5, label=f'r_nom = {r_nom:.2f} cm')
    ax1.axvline(r_test, color='orange', linestyle=':', linewidth=1.5, label=f'r_test = {r_test:.2f} cm')
    ax1.axhspan(1.00, 1.15, color='green', alpha=0.12, label='Expected range [1.00, 1.15]')
    ax1.axhline(1.0, color='red', linestyle='--', alpha=0.4, linewidth=1)

    stats_str5 = (f'r_nom = {r_nom:.3f} cm\n'
                  f'd(rho)/d(r) = {sens:.4f} g/cm3 per cm\n'
                  f'Error multiplier: {rel_sens:.0f}x  (1% in r -> 2% in rho)')
    ax1.text(0.02, 0.97, stats_str5, transform=ax1.transAxes, fontsize=8,
             va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    ax1.set_xlabel('Vessel radius r (cm)', fontsize=12)
    ax1.set_ylabel('Density rho (g/cm3)', fontsize=12)
    ax1.set_title('Density sensitivity to vessel radius\n(each curve = one specimen)', fontsize=12)
    ax1.legend(fontsize=8, framealpha=0.9)

    # --- Right: original vs recalculated density at r_test ---
    sub5['rho_recalc'] = sub5['DENS_mass_g'] / (np.pi * r_test ** 2 * sub5['DENS_delta_h_cm'])

    data_orig   = [sub5[(sub5['pva_real'] == r.pva_real) & (sub5['pg_pct'] == r.pg_pct)]['DENS_density_gcm3'].dropna().values for _, r in conditions.iterrows()]
    data_recalc = [sub5[(sub5['pva_real'] == r.pva_real) & (sub5['pg_pct'] == r.pg_pct)]['rho_recalc'].dropna().values         for _, r in conditions.iterrows()]

    x_pos = np.arange(len(conditions))
    w = 0.35
    bp_o = ax2.boxplot(data_orig,   positions=x_pos - w / 2, widths=w * 0.85,
                        patch_artist=True, manage_ticks=False)
    bp_r = ax2.boxplot(data_recalc, positions=x_pos + w / 2, widths=w * 0.85,
                        patch_artist=True, manage_ticks=False)
    for patch in bp_o['boxes']:
        patch.set_facecolor('steelblue'); patch.set_alpha(0.6)
    for patch in bp_r['boxes']:
        patch.set_facecolor('orange');   patch.set_alpha(0.6)

    ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax2.axhspan(1.00, 1.15, color='green', alpha=0.08)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(labels_cond, fontsize=7)
    ax2.set_ylabel('Density (g/cm3)', fontsize=12)
    ax2.set_title(f'Original (r={r_nom:.2f}) vs Recalculated (r={r_test:.2f})', fontsize=12)
    ax2.legend([bp_o['boxes'][0], bp_r['boxes'][0]],
               [f'r = {r_nom:.2f} cm', f'r = {r_test:.2f} cm'], fontsize=9)

    plt.tight_layout()
    plt.savefig(fig_dir / 'B3_05_density_radius_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()

    pct_below = (sub5['DENS_density_gcm3'] < 1.0).mean() * 100
    dr = abs(r_test - r_nom)
    drho_pct = 2 * dr / r_nom * 100
    print(f'Nominal radius: {r_nom:.3f} cm')
    print(f'Test radius:    {r_test:.3f} cm  (delta r = {dr:.3f} cm = {dr/r_nom*100:.1f}%)')
    print(f'Expected density shift: ~{drho_pct:.1f}% (from -2*dr/r rule)')
    print(f'Mean rho (nominal): {sub5["DENS_density_gcm3"].mean():.4f} g/cm3')
    print(f'Mean rho (r_test):  {sub5["rho_recalc"].mean():.4f} g/cm3')
    print(f'Density values < 1.0 (nominal r): {pct_below:.1f}%')


---
## Cell 3.6 — Density validation: mass and displaced volume

Left: sample mass (g) by condition. Right: displaced volume $V_{\text{disp}}$ (cm³) by condition.
Same-composition pieces should cluster tightly; outliers suggest weighing error or piece damage.

**Red flags:** suspiciously round masses (manual entry error); $V_{\text{disp}} \leq 0$;
$V_{\text{disp}} \gg$ mass (density << 1, physically unreasonable).

In [ ]:
# --- Cell 3.6: Density validation — mass and volume ---
rng6 = np.random.default_rng(46)

has_mass = 'DENS_mass_g'    in df.columns and df['DENS_mass_g'].notna().any()
has_vdisp = 'DENS_V_disp_cm3' in df.columns and df['DENS_V_disp_cm3'].notna().any()

if not has_mass and not has_vdisp:
    print('DENS_mass_g and DENS_V_disp_cm3 not available.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, col, ylabel, title in [
        (axes[0], 'DENS_mass_g',     'Mass (g)',        'Sample mass by condition'),
        (axes[1], 'DENS_V_disp_cm3', 'V_disp (cm3)',    'Displaced volume by condition'),
    ]:
        if col not in df.columns or not df[col].notna().any():
            ax.text(0.5, 0.5, f'{col} not available', ha='center', transform=ax.transAxes)
            continue

        data_val = [
            df[(df['pva_real'] == r.pva_real) & (df['pg_pct'] == r.pg_pct)][col].dropna().values
            for _, r in conditions.iterrows()
        ]
        bp = ax.boxplot(data_val, labels=labels_cond, patch_artist=True,
                        medianprops={'color': 'black', 'linewidth': 2},
                        whiskerprops={'linewidth': 1.2}, capprops={'linewidth': 1.2})
        for i, (patch, col_c) in enumerate(zip(bp['boxes'], colors_box)):
            patch.set_facecolor(col_c)
            patch.set_alpha(0.5)
            if len(data_val[i]) > 0:
                x_pts = rng6.uniform(-0.15, 0.15, size=len(data_val[i])) + (i + 1)
                ax.scatter(x_pts, data_val[i], color=col_c, s=30, zorder=5,
                           edgecolors='white', linewidths=0.5, alpha=0.9)

        ax.set_xlabel('Condition', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(title + '\n(same PVA → similar mass expected)', fontsize=12)
        ax.tick_params(axis='x', labelsize=8)

    plt.tight_layout()
    plt.savefig(fig_dir / 'B3_06_mass_vdisp.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Red flags
    print('Red flags:')
    found6 = False
    if has_vdisp:
        bad_vdisp = df[df['DENS_V_disp_cm3'] <= 0][['pva_real', 'pg_pct', 'piece', 'cycle', 'DENS_V_disp_cm3']]
        if not bad_vdisp.empty:
            print(f'  [!] {len(bad_vdisp)} row(s) with V_disp <= 0:')
            display(bad_vdisp)
            found6 = True
    if has_mass:
        # Check for suspiciously round masses (integer or .5 values)
        m_vals = df['DENS_mass_g'].dropna()
        n_round = int((m_vals == m_vals.round(0)).sum())
        if n_round > len(m_vals) * 0.3:
            print(f'  [!] {n_round}/{len(m_vals)} mass values are integers — possible manual entry approximation.')
            found6 = True
    if not found6:
        print('  None detected.')


---
## Cell 3.7 — Outlier detection (IQR method)

Within each (PVA%, PG%, cycle) group: flag values outside $Q_1 - 1.5\,\text{IQR}$ and
$Q_3 + 1.5\,\text{IQR}$.

> **With n=5 the IQR method is crude.** A flagged point is not necessarily wrong — it is
> a point worth investigating. Never delete data without justification; always report
> analysis both with and without suspect points if they are excluded (*Study Rules §8*).

In [ ]:
# --- Cell 3.7: Outlier detection — IQR method per group ---
check_vars = [
    ('US_Cl',             'Cl',      'm/s',    (1480, 1700)),
    ('DENS_density_gcm3', 'density', 'g/cm3',  (0.95, 1.20)),
    ('US_d',              'd',       'm',       (0.005, 0.05)),
    ('US_T_mean',         'T_mean',  'degC',   (30, 40)),
]
check_vars = [(c, lbl, u, r) for c, lbl, u, r in check_vars if c in df.columns and df[c].notna().any()]

outlier_rows = []
for (pva, pg, cyc), grp in df.groupby(['pva_real', 'pg_pct', 'cycle']):
    for col, lbl, unit, phys_range in check_vars:
        vals = grp[col].dropna()
        if len(vals) < 3:
            continue
        Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
        IQR = Q3 - Q1
        lo_fence = Q1 - 1.5 * IQR
        hi_fence = Q3 + 1.5 * IQR
        out_mask = (vals < lo_fence) | (vals > hi_fence)
        # Also flag values outside physical plausible range
        phys_mask = (vals < phys_range[0]) | (vals > phys_range[1])
        flag_mask = out_mask | phys_mask
        for idx in vals[flag_mask].index:
            val = vals[idx]
            flags = []
            if out_mask.get(idx, False):
                flags.append('IQR')
            if phys_mask.get(idx, False):
                flags.append('physical')
            piece_val = grp.loc[idx, 'piece'] if 'piece' in grp.columns else '?'
            outlier_rows.append({
                'PVA (%)':     pva,
                'PG (%)':      int(pg),
                'Cycle':       int(cyc),
                'Piece':       piece_val,
                'Variable':    lbl,
                'Unit':        unit,
                'Value':       round(val, 5),
                'Lo fence':    round(lo_fence, 5),
                'Hi fence':    round(hi_fence, 5),
                'Flag':        '+'.join(flags),
            })

if outlier_rows:
    df_outliers = pd.DataFrame(outlier_rows)
    print(f'{len(df_outliers)} flagged value(s):\n')
    display(df_outliers.set_index(['PVA (%)', 'PG (%)', 'Cycle', 'Piece']))

    print('\nNext steps for each flagged point:')
    print('  1. Load raw signals (ecos_loader.load_signals) and visually inspect')
    print('  2. Check whether the same piece is flagged in other variables or cycles')
    print('  3. If consistently anomalous -> fabrication defect (document, possibly exclude)')
    print('  4. If isolated -> measurement error (document, consider remeasuring)')
    print('  5. If excluded: always report analysis WITH and WITHOUT the excluded point')

    # How many unique pieces are flagged?
    n_pieces = df_outliers[['PVA (%)', 'PG (%)', 'Piece']].drop_duplicates().shape[0]
    print(f'\nUnique pieces flagged: {n_pieces}')
    piece_counts = df_outliers.groupby(['PVA (%)', 'PG (%)', 'Piece'])['Variable'].count()
    if (piece_counts > 1).any():
        print('Pieces flagged in multiple variables (more likely a genuine problem):')
        display(piece_counts[piece_counts > 1].reset_index().rename(columns={'Variable': 'n_flags'}))
else:
    print('No outliers detected by IQR or physical plausibility criteria.')


---
## Cell 3.8 — Reproducibility: coefficient of variation (CV%) heatmaps

CV = SD/mean × 100% per condition. Computed per (PVA%, PG%, cycle) then averaged
across cycles. Left: CV for $C_l$. Right: CV for density.

**Thresholds (Study Rules §5):** CV < 1% for $C_l$ = excellent; 1–3% = acceptable;
> 5% = poor. IEC 60601 for tissue-mimicking phantoms typically requires CV < 5%.

In [ ]:
# --- Cell 3.8: CV heatmaps — reproducibility per condition ---
def compute_cv_heatmap(df, col, pva_idx, pg_idx):
    mat = np.full((len(pva_idx), len(pg_idx)), np.nan)
    n_mat = np.full_like(mat, np.nan)
    for (pva, pg, cyc), grp in df.groupby(['pva_real', 'pg_pct', 'cycle']):
        vals = grp[col].dropna()
        if len(vals) < 2 or pva not in pva_idx or pg not in pg_idx:
            continue
        m = vals.mean()
        cv = vals.std(ddof=1) / m * 100 if m != 0 else np.nan
        i, j = pva_idx.index(pva), pg_idx.index(pg)
        if np.isnan(mat[i, j]):
            mat[i, j]   = cv
            n_mat[i, j] = 1
        else:
            mat[i, j]   = (mat[i, j] * n_mat[i, j] + cv) / (n_mat[i, j] + 1)
            n_mat[i, j] += 1
    return mat

pva_idx = sorted(df['pva_real'].dropna().unique())
pg_idx  = sorted(df['pg_pct'].dropna().unique())

cv_cl   = compute_cv_heatmap(df, 'US_Cl',             pva_idx, pg_idx)
cv_dens = compute_cv_heatmap(df, 'DENS_density_gcm3', pva_idx, pg_idx)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

for ax, mat, title, cmap_name in [
    (ax1, cv_cl,   'CV% for $C_l$ (averaged over cycles)',      'RdYlGn_r'),
    (ax2, cv_dens, 'CV% for density (averaged over cycles)', 'RdYlGn_r'),
]:
    vmax = max(5.0, np.nanmax(mat)) if not np.all(np.isnan(mat)) else 5.0
    im = ax.imshow(mat, aspect='auto', cmap=cmap_name, vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label='CV (%)')
    ax.set_xticks(range(len(pg_idx)))
    ax.set_yticks(range(len(pva_idx)))
    ax.set_xticklabels([f'PG {int(v)}%' for v in pg_idx], fontsize=10)
    ax.set_yticklabels([f'PVA {v:.4g}%' for v in pva_idx], fontsize=10)
    ax.set_xlabel('PG concentration (%)', fontsize=12)
    ax.set_ylabel('PVA concentration (%)', fontsize=12)
    ax.set_title(title, fontsize=12)
    for i in range(len(pva_idx)):
        for j in range(len(pg_idx)):
            if not np.isnan(mat[i, j]):
                txt = f'{mat[i,j]:.2f}%'
                color = 'white' if mat[i, j] > vmax * 0.7 else 'black'
                ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=color)

plt.tight_layout()
plt.savefig(fig_dir / 'B3_08_CV_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Cl   CV% — min: {np.nanmin(cv_cl):.2f}%  max: {np.nanmax(cv_cl):.2f}%  mean: {np.nanmean(cv_cl):.2f}%')
print(f'Dens CV% — min: {np.nanmin(cv_dens):.2f}%  max: {np.nanmax(cv_dens):.2f}%  mean: {np.nanmean(cv_dens):.2f}%')
if np.nanmax(cv_cl) > 5:
    print('[!] Some conditions show CV(Cl) > 5% — investigate fabrication or measurement error.')
if np.nanmax(cv_dens) > 5:
    print('[!] Some conditions show CV(density) > 5% — investigate fabrication consistency.')


---
## Cell 3.9 — Residual analysis: variance decomposition

Full factorial ANOVA: $C_l \sim$ PVA% + PG% + Cycle + PVA%:PG% + Temperature.
The $\eta^2$ for each term shows what fraction of total $C_l$ variance it explains.

**Ideal:** PVA and PG account for most variance; temperature < 5%; small residual.
**Concerning:** large residual → important uncontrolled sources of variability;
temperature > 5% → confounding concern.

Requires `statsmodels`; falls back to one-way ANOVA breakdown if unavailable.

In [ ]:
# --- Cell 3.9: Variance decomposition — Cl ~ PVA + PG + cycle + PVA:PG + T ---
anova_cols = ['US_Cl', 'pva_real', 'pg_pct', 'cycle']
has_T      = df['US_T_mean'].notna().any()
if has_T:
    anova_cols.append('US_T_mean')

df_av = df[anova_cols].dropna().copy()
df_av['pva_cat']   = df_av['pva_real'].astype(str)
df_av['pg_cat']    = df_av['pg_pct'].astype(str)
df_av['cycle_cat'] = df_av['cycle'].astype(str)
print(f'Full model sample size: {len(df_av)}')

try:
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import anova_lm

    formula = 'US_Cl ~ C(pva_cat) + C(pg_cat) + C(cycle_cat) + C(pva_cat):C(pg_cat)'
    if has_T:
        formula += ' + US_T_mean'

    model_full = smf.ols(formula, data=df_av).fit()
    aov_full   = anova_lm(model_full, typ=2)
    SS_total   = aov_full['sum_sq'].sum()
    aov_full['eta2']   = aov_full['sum_sq'] / SS_total
    aov_full['pct_var'] = aov_full['eta2'] * 100

    label_map = {
        'C(pva_cat)':              'PVA%',
        'C(pg_cat)':               'PG%',
        'C(cycle_cat)':            'Cycle',
        'C(pva_cat):C(pg_cat)':    'PVA% x PG%',
        'US_T_mean':               'Temperature',
        'Residual':                'Residual',
    }
    aov_full.index = [label_map.get(idx, idx) for idx in aov_full.index]

    disp9 = aov_full[['sum_sq', 'df', 'F', 'PR(>F)', 'pct_var']].copy()
    disp9.columns = ['SS', 'df', 'F', 'p-value', '% variance']
    print('\nVariance decomposition (Type II SS, full model):')
    display(disp9.style.format({'SS': '{:.2f}', 'df': '{:.0f}', 'F': '{:.3f}',
                                 'p-value': '{:.4f}', '% variance': '{:.2f}'}))

    # Bar chart of variance fractions
    sources = disp9.index.tolist()
    pct_vals = disp9['% variance'].values
    bar_colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple', 'goldenrod', 'lightgrey']
    bar_colors = (bar_colors * 3)[:len(sources)]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(sources, pct_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, pct_vals):
        ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('% of total Cl variance', fontsize=12)
    ax.set_title('Variance decomposition: Cl ~ PVA + PG + Cycle + PVAxPG + T', fontsize=12)
    ax.set_xlim(0, max(pct_vals) * 1.15)
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
    ax.grid(False, axis='y')
    plt.tight_layout()
    plt.savefig(fig_dir / 'B3_09_variance_decomposition.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Flag temperature contribution
    if 'Temperature' in disp9.index:
        t_pct = disp9.loc['Temperature', '% variance']
        if t_pct > 5:
            print(f'[!] Temperature explains {t_pct:.1f}% of Cl variance — confounding concern.')
        else:
            print(f'Temperature explains {t_pct:.1f}% of Cl variance — acceptable.')
    resid_pct = disp9.loc['Residual', '% variance'] if 'Residual' in disp9.index else np.nan
    if not np.isnan(resid_pct) and resid_pct > 30:
        print(f'[!] Residual = {resid_pct:.1f}% — substantial unexplained variance. Missing factor?')

except ImportError:
    print('statsmodels not available. Running one-way ANOVA per factor instead.')
    for label, col_cat in [('PVA%', 'pva_cat'), ('PG%', 'pg_cat'), ('Cycle', 'cycle_cat')]:
        groups_v = [g['US_Cl'].dropna().values for _, g in df_av.groupby(col_cat)]
        if len(groups_v) >= 2 and all(len(g) >= 2 for g in groups_v):
            F, p = stats.f_oneway(*groups_v)
            # Compute eta2 manually
            all_vals = np.concatenate(groups_v)
            grand_mean = all_vals.mean()
            SS_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups_v)
            SS_total_v = sum((v - grand_mean)**2 for g in groups_v for v in g)
            eta2 = SS_between / SS_total_v if SS_total_v > 0 else np.nan
            print(f'  {label}: F={F:.3f}, p={p:.4f}, eta2={eta2:.4f} ({eta2*100:.1f}% variance)')
    print('Install statsmodels for the full multi-factor model: pip install statsmodels')


---
## Cell 3.10 — Summary and interpretation

> **Instructions:** Fill in this template after running all cells above.
> This summary should accompany the Methods section of the paper as
> evidence of measurement quality.

---

### Block 3 — Experimental Control Summary

**Temperature control (US):**
- Mean ± SD: *[X]* ± *[Y]* °C (target: 36°C) — from Cell 3.0
- Range: *[min]*–*[max]* °C
- % within ±1°C of target: *[Z]*%
- Confounding with $C_l$: r = *[value]*, p = *[value]* — Cell 3.2
- Confounding with condition: *[yes/no, from Cell 3.3 ANOVA]*
- Confounding with cycle: *[yes/no, from Cell 3.4 ANOVA]*

**Sensor agreement (T1 vs T2):**
- Mean T1 − T2: *[X]* ± *[Y]* °C
- Limits of agreement: *[[lo, hi]]* °C
- Significant bias? *[yes/no, t-test p-value]*
- Estimated $C_w$ bias: *[~X m/s]*

**Density measurement:**
- Nominal vessel radius: *[r_nom]* cm
- Error multiplier: $d\rho/\rho = -2\,dr/r$ (1% in r → 2% in $\rho$)
- Mean density (nominal r): *[X]* g/cm³
- Values below 1.0 g/cm³: *[count]* out of *[total]*
- Impact of r_test: *[describe shift in mean density]*

**Outliers identified (Cell 3.7):**
- *[list, or "none detected"]*
- Pieces flagged in multiple variables: *[list or "none"]*

**Reproducibility (Cell 3.8):**
- CV for $C_l$: range *[min]*–*[max]*% (mean *[X]*%) across conditions
- CV for density: range *[min]*–*[max]*%
- Assessment: *[excellent (CV<1%) / acceptable (1–3%) / concerning (>5%)]*

**Variance decomposition (Cell 3.9):**
- PVA% explains *[X]*% of $C_l$ variance
- PG% explains *[Y]*%
- Cycle explains *[Z]*%
- Temperature explains *[W]*%  (*[confound / not confound]*)
- Unexplained (residual): *[R]*%

**Overall measurement quality assessment:**
- *[Are the measurements trustworthy? Which caveats must be stated in the paper?]*
- *[List any systematic errors identified and their estimated magnitude.]*
- *[List any recommended follow-up measurements or calibrations.]*

---
*Study Rules §4 (systematic errors) and §6 (reporting standards) apply throughout this block.*